# 🚀 End-to-End Extraction, Segmentation, and AI Validation Pipeline
This notebook executes the entire zero-shot pipeline (Grounding DINO + SAM 2) and introduces a novel **VLM-as-a-Judge (Qwen3-VL)** step. 

Instead of manual QA, we pass the resulting segmentation masks back into the Vision Language Model and ask it to critique the segmentation quality, identifying background leakage, missing feathers, or clipped boundaries.

In [ ]:
# =====================================================================
# STEP 1: SETTING UP THE DIGITAL LABORATORY
# =====================================================================
# Think of this cell like gathering all your tools before starting an experiment.
# We are bringing in Python "libraries" (pre-written code) that let us:
# - Read image files (cv2, PIL)
# - Do heavy math (numpy, torch)
# - Run Artificial Intelligence models (transformers, ultralytics, mlx_vlm)

import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import SAM
from mlx_vlm import load, generate

# We tell the computer to use the Mac's powerful "M-series" chip (MPS) instead of the basic processor (CPU).
# This makes the AI run much faster.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Hardware Accelerator: {device}")

# Tell the code where to find our pictures (raw) and where to save the results (processed)
DATA_DIR = "../data/raw"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True) # Create the output folder if it doesn't exist yet

# Select the picture we want to test
image_path = os.path.join(DATA_DIR, 'A1383 2000-im1316.jpg')

In [ ]:
# =====================================================================
# STEP 2: FINDING AND TRACING THE FEATHERS (THE AI PIPELINE)
# =====================================================================
# This function does two main things:
# 1. "DINO" (The Finder): We give it text ("bird feather") and it draws a rough box around where it thinks they are.
# 2. "SAM 2" (The Tracer): We give it DINO's boxes, and it acts like a digital scalpel, cutting out the exact pixel shape of the feather inside that box.

def run_pipeline(img_path):
    print("1. Waking up Grounding DINO (The Finder AI)...")
    dino_id = 'IDEA-Research/grounding-dino-base'
    dino_processor = AutoProcessor.from_pretrained(dino_id)
    dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)
    
    # Open the picture so the AI can look at it
    img = Image.open(img_path).convert('RGB')
    
    # We explicitly ask the AI to find "bird feather."
    inputs = dino_processor(images=img, text='bird feather.', return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = dino_model(**inputs) # The AI looks at the image here!
    
    # Clean up the AI's output into a list of boxes.
    results = dino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, target_sizes=[img.size[::-1]]
    )[0]
    
    # Only keep the boxes where the AI is more than 45% sure it found a feather.
    # NOTE: If this is too low, it finds fake feathers. If it's too high, it misses real ones!
    boxes = [box.tolist() for score, box in zip(results['scores'], results['boxes']) if score > 0.45]
    
    # Put DINO to sleep to save computer memory
    del dino_model
    torch.mps.empty_cache()
    
    print(f"   Found {len(boxes)} boxes! Passing them to SAM 2 (The Tracer AI)...")
    
    # Wake up SAM 2 and hand it the picture and the rough boxes.
    # It will return highly detailed "masks" (silhouettes) of the objects.
    sam_model = SAM('sam2.1_b.pt')
    sam_results = sam_model(img_path, bboxes=boxes, device='mps', verbose=False)[0]
    
    return boxes, sam_results

boxes, sam_results = run_pipeline(image_path)

In [ ]:
# =====================================================================
# STEP 3: DRAWING THE RESULTS FOR REVIEW
# =====================================================================
# Now we take the original picture and draw the AI's work on top of it.
# - Bright Green Boxes = Where DINO looked.
# - Red Paint = What SAM 2 decided was the actual feather.
# This makes it easy for us (and the final AI Judge) to see if mistakes were made.

def create_validation_composite(img_path, boxes, sam_res):
    # Read the image and create a copy we can draw on (like a transparent plastic sheet)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    overlay = img_rgb.copy()
    
    # Paint the exact shape of the feathers in RED
    if sam_res.masks is not None:
        for m in sam_res.masks.data.cpu().numpy():
            m = cv2.resize(m.astype(np.float32), (img.shape[1], img.shape[0]))
            overlay[m > 0.5] = [255, 0, 0] # [Red, Green, Blue] -> 255 Red means pure red!
            
    # Draw the rough bounding boxes in BRIGHT GREEN
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 6) # 0, 255, 0 is pure green.
        
    # Blend the transparent sheet (with our drawings) over the original image
    blended = cv2.addWeighted(img_rgb, 0.4, overlay, 0.6, 0)
    
    # Save this new drawing to our hard drive
    comp_path = os.path.join(OUTPUT_DIR, "validation_composite.jpg")
    cv2.imwrite(comp_path, cv2.cvtColor(blended, cv2.COLOR_RGB2BGR))
    
    # Show the drawing right here on the screen
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(blended)
    ax.set_title("AI Quality Assurance Composite (Masks & Bounds)")
    ax.axis('off')
    plt.show()
    
    return comp_path

composite_path = create_validation_composite(image_path, boxes, sam_results)

In [ ]:
# =====================================================================
# STEP 4: THE AI QUALITY CONTROL INSPECTOR (QWEN3-VL)
# =====================================================================
# Instead of a human checking thousands of pictures manually, we ask a third,
# very smart "Vision Language" AI (Qwen3-VL) to look at the drawing we just made.
# We give it specific instructions on what to look for (missing feathers, paint spilling over)
# and ask it to grade the work out of 10.

def vlm_as_a_judge(comp_img_path):
    print("Loading Qwen3-VL to judge segmentation quality...")
    vlm_path = "mlx-community/Qwen3-VL-8B-Instruct-4bit"
    model, processor = load(vlm_path)
    
    # The instructions we are giving to the AI Inspector
    raw_prompt = '''
    You are an expert computer vision Quality Assurance bot. 
    Analyze the provided image. The bright green boxes represent bounding box detections. The red shaded areas represent pixel-perfect segmentation masks generated by SAM 2.
    
    Critique the quality of the segmentation. 
    1. Are the red masks covering the bird feathers entirely? Look closely, did it miss the middle feather?
    2. Is there background leakage (red mask bleeding onto the white paper, ruler, or tape)?
    3. Are the green boxes wrapping multiple feathers instead of individual ones?
    4. Provide an overall confidence score out of 10.
    
    Return EXACTLY in this JSON format:
    {
      "all_feathers_covered": true/false,
      "background_leakage_detected": true/false,
      "green_boxes_grouped_feathers": true/false,
      "quality_score_1_to_10": 9,
      "critique": "Your brief explanation here."
    }
    '''
    
    print("Asking VLM to critique the composite...")
    try:
        # Wrap our instructions in a special "ChatML" format so the AI understands it's a conversation
        chatml_prompt = f"<|im_start|>system\nYou are a highly critical and observant AI assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>\n{raw_prompt}<|im_end|>\n<|im_start|>assistant\n"
        
        # Ask the AI to generate an answer
        output = generate(model, processor, prompt=chatml_prompt, image=[comp_img_path], max_tokens=256)
        
        output_text = getattr(output, 'text', str(output))
        
        # We use a tool called "Regex" to search the AI's text and only pull out the dictionary format (JSON).
        import re
        match = re.search(r'\{.*?\}', output_text, re.DOTALL)
        
        print("\n=== VLM JUDGE REPORT ===")
        if match:
            print(match.group(0)) # Print the clean JSON dictionary
        else:
            print(output_text)    # If it failed to write a dictionary, just print what it said
    except Exception as e:
        print(f"VLM evaluation failed: {e}")

vlm_as_a_judge(composite_path)